# Reference Wavefront: FAM Intrinsics Plots (v2)

**Author:** Aaron Roodman  
**Date Created:** 2026-02-23  
**Last Modified:** 2026-03-15  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Full Array Mode, Zernike, Analysis

## Description

This notebook analyzes the FAM Zernike table created by `intrinsics_mktable.ipynb`.

**Analysis includes:**
1. Load parquet file with Zernike measurements
2. Fit per-image linear model (constant + x,y slopes) to data-model residuals
3. Plot fit parameters vs image for Z4-Z15
4. Create hexbin visualizations: Data, Model, and Data-Model residuals

**Input:** Parquet file created by mktable notebook  
**Output:** Trio comparison plots (PNG), fit parameter plots

**Based on:** `intrinsics_mktable.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-15 | Aaron Roodman | Added CCS/OCS coord_sys parameter; added per-image linear fit (constant + x,y slopes) replacing simple mean subtraction; added fit parameter plots vs image for Z4-Z15; updated trio plots to use linear fit subtraction; added rotator angle filter (default -3 to 3 deg) |
| 2026-03-15 | Aaron Roodman | Fixed iZs: infer Zernike indices from data shape instead of hardcoding (table has Z4-Z22 contiguous, 19 terms) |
| 2026-03-15 | Aaron Roodman | Added day_obs_plot parameter for selecting subset of days for plots; fixed k2/k3 units to μm/deg; added fit residual histograms (log scale, ±1 μm) for Z4-Z15 to study outliers |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Load Data](#load)
5. [Analysis](#analysis)
6. [Linear Fit](#fit)
7. [Plot Selection](#fitplots)
8. [Fit Parameter Plots](#fitplots)
9. [Fit Residual Histograms](#residhist)
10. [Visualizations](#viz)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system: 'CCS' or 'OCS' (must match mktable choice)
coord_sys = 'OCS'

# Parquet file dates
day_obs_min = 20250801
day_obs_max = 20251028

# Date range string for plot titles
date_range_str = f'{day_obs_min}-{day_obs_max}'

# Rotator angle filter (degrees) — only keep images within this range
rotator_angle_min = -3.0
rotator_angle_max = 3.0

# Zernike indices: inferred from data after loading (see Load Data section)
# iZs and iZidx are set dynamically below

# Plotting parameters
plo_default = 4.0   # Low percentile for colorbar
phi_default = 96.0  # High percentile for colorbar
output_dir = 'output'

# Ensure output directory exists
Path(output_dir).mkdir(parents=True, exist_ok=True)

<a id='setup'></a>
## Setup & Imports

In [1]:
# Standard imports
from matplotlib import pyplot as plt
from matplotlib import lines
from mpl_toolkits import axes_grid1
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import pandas as pd
from scipy.stats import binned_statistic_2d, binned_statistic
from pathlib import Path

# Astropy
from astropy.table import Table, QTable

# Set plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Pandas display options
pd.options.display.max_columns = None
pd.options.display.width = None

<a id='load'></a>
## Load Data

In [ ]:
# Load the parquet file
parquet_file = f'fam_zernikes_{day_obs_min}_{day_obs_max}.parquet'

if not Path(parquet_file).exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_file}\n"
                           f"Run intrinsics_mktable.ipynb first to create it.")

print(f"Loading data from: {parquet_file}")
aosTable = QTable.read(parquet_file)
print(f"Loaded {len(aosTable)} rows")
print(f"Columns: {len(aosTable.columns)}")
print(f"Coordinate system: {coord_sys}")

In [3]:
# Display table summary
print("\nTable columns:")
print(sorted(aosTable.columns))

# Count by day_obs
print("\nMeasurements per day_obs:")
day_counts = pd.DataFrame({'day_obs': aosTable['day_obs']}).value_counts().sort_index()
print(day_counts)


Table columns:
['centroid_x', 'centroid_x_extra', 'centroid_x_intra', 'centroid_y', 'centroid_y_extra', 'centroid_y_intra', 'coord_dec', 'coord_dec_extra', 'coord_dec_intra', 'coord_ra', 'coord_ra_extra', 'coord_ra_intra', 'day_obs', 'detector', 'extra_fpx', 'extra_fpy', 'intra_fpx', 'intra_fpy', 'matched_intra_extra', 'seq_num', 'th_N', 'th_N_extra', 'th_N_intra', 'th_W', 'th_W_extra', 'th_W_intra', 'thx_CCS', 'thx_CCS_extra', 'thx_CCS_intra', 'thx_OCS', 'thx_OCS_extra', 'thx_OCS_intra', 'thy_CCS', 'thy_CCS_extra', 'thy_CCS_intra', 'thy_OCS', 'thy_OCS_extra', 'thy_OCS_intra', 'used', 'zk_CCS', 'zk_CCS_mean', 'zk_NW', 'zk_OCS', 'zk_intrinsic', 'zk_residual']

Measurements per day_obs:
day_obs 
20250825    120458
20250826     94567
20250907      1586
20250909      7029
20250912     11256
20250913     52244
20251023     44375
20251024      9162
20251026     79514
20251027     20498
20251028      7374
Name: count, dtype: int64


In [4]:
# Display sample rows with formatted floats
# Select scalar columns only
scalar_cols = []
for col in aosTable.columns:
    if not hasattr(aosTable[col][0], '__len__') or isinstance(aosTable[col][0], str):
        scalar_cols.append(col)

df_display = aosTable[scalar_cols].to_pandas()
pd.options.display.float_format = '{:.2f}'.format

print("\nSample rows (first 5):")
print(df_display.head())

pd.reset_option('display.float_format')


Sample rows (first 5):
  detector  used  coord_ra  coord_dec  centroid_x  centroid_y  thx_CCS  \
0  R01_S00  True      4.72      -0.38     3773.04     3752.49    -0.03   
1  R01_S01  True      4.72      -0.38     1991.94     3338.47    -0.03   
2  R01_S01  True      4.72      -0.38     2997.63     2987.60    -0.03   
3  R01_S01  True      4.72      -0.38     2273.00     2673.47    -0.03   
4  R01_S01  True      4.72      -0.38      988.99     2905.48    -0.03   

   thy_CCS  thx_OCS  thy_OCS  th_N  th_W  coord_ra_intra  coord_ra_extra  \
0    -0.01    -0.03    -0.01  0.03 -0.02            4.72            4.72   
1    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
2    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
3    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
4    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   

   coord_dec_intra  coord_dec_extra  centroid_x_intra  centroid_x_extra  \

In [ ]:
# Filter by rotator angle
if 'rotator_angle' in aosTable.columns:
    rot_angles = np.array(aosTable['rotator_angle'])
    rot_mask = (rot_angles >= rotator_angle_min) & (rot_angles <= rotator_angle_max)
    n_before = len(aosTable)
    aosTable = aosTable[rot_mask]
    n_after = len(aosTable)
    n_images_before = len(set(zip(np.array(aosTable['day_obs']), np.array(aosTable['seq_num']))))
    print(f"Rotator angle filter [{rotator_angle_min}, {rotator_angle_max}] deg:")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
    print(f"  Remaining images: {n_images_before}")
else:
    print("Warning: 'rotator_angle' column not found in table.")
    print("  Rotator angle filter NOT applied.")
    print("  Re-run intrinsics_mktable.ipynb to include rotator angles.")

<a id='functions'></a>
## Helper Functions

In [ ]:
def get_zernike(table, column_name, iZ):
    """Extract a single Zernike term from an array column.
    
    Parameters
    ----------
    table : QTable
        Table with Zernike array columns
    column_name : str
        Column name (e.g. 'zk_OCS', 'zk_intrinsic', 'zk_residual')
    iZ : int
        Zernike index (4-28, excluding 20, 21)
    
    Returns
    -------
    array : ndarray
        Zernike values
    """
    if iZ not in iZidx:
        raise ValueError(f"Zernike Z{iZ} not in table. Available: {iZs}")
    
    zk_array = np.stack(table[column_name])
    return zk_array[:, iZidx[iZ]]


def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1./aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


print("Helper functions loaded: get_zernike(), add_colorbar()")
print(f"\nField angle columns: thx_{coord_sys}, thy_{coord_sys} (radians)")
print(f"Zernike data column: zk_{coord_sys}")

In [ ]:
# Extract Zernike arrays and infer Zernike indices from data shape
zk_data = np.stack(aosTable[f'zk_{coord_sys}'])
zk_model = np.stack(aosTable['zk_intrinsic'])

nZk = zk_data.shape[1]

# Infer Zernike indices (must match infer_zernike_indices in mktable)
if nZk == 19:
    iZs = list(range(4, 23))   # Z4 through Z22 (contiguous)
else:
    iZs = list(range(4, 4 + nZk))
    print(f"Warning: Unexpected number of Zernike terms ({nZk}), assuming Z4-Z{3 + nZk}")

iZidx = {iZ: i for i, iZ in enumerate(iZs)}

print(f"Extracted Zernike arrays:")
print(f"  Shape: {zk_data.shape} (rows x Zernikes)")
print(f"  Zernike indices: {iZs}")
print(f"  Coordinate system: {coord_sys}")

<a id='analysis'></a>
## Analysis

Filter for matched intra/extra donuts, then fit per-image linear model and create trio plots.

In [8]:
# Filter for matched intra/extra donuts
matched_mask = aosTable['matched_intra_extra']
print(f"Total donuts: {len(aosTable)}")
print(f"Matched intra/extra donuts: {np.sum(matched_mask)}")
print(f"Fraction matched: {np.sum(matched_mask)/len(aosTable):.3f}")

# Apply filter
aosTable_matched = aosTable[matched_mask]

Total donuts: 448063
Matched intra/extra donuts: 323608
Fraction matched: 0.722


<a id='fit'></a>
## Linear Fit per Image

For each image and Zernike term, fit: `zJ_data = zJ_model + k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3`  
where `Zfocal_k2 = 2.0 * thx` and `Zfocal_k3 = 2.0 * thy`.

In [ ]:
# Compute linear fit per image: (data - model) = k1 + k2 * 2*thx + k3 * 2*thy
# Convert field angles to degrees so slope units are μm/deg
thx_col = f'thx_{coord_sys}'
thy_col = f'thy_{coord_sys}'

# Get arrays from matched table (convert radians to degrees)
day_obs_arr = np.array(aosTable_matched['day_obs'])
seq_num_arr = np.array(aosTable_matched['seq_num'])
thx_arr = np.rad2deg(np.array(aosTable_matched[thx_col]))
thy_arr = np.rad2deg(np.array(aosTable_matched[thy_col]))
zk_data_matched = np.stack(aosTable_matched[f'zk_{coord_sys}'])
zk_model_matched = np.stack(aosTable_matched['zk_intrinsic'])

# Identify unique images
images = sorted(set(zip(day_obs_arr.tolist(), seq_num_arr.tolist())))
n_donuts = len(aosTable_matched)
n_zernikes = len(iZs)

# Storage for per-donut fitted values and per-image fit parameters
zk_fit = np.zeros((n_donuts, n_zernikes))
fit_params_list = []

for img_idx, (dobs, snum) in enumerate(images):
    mask = (day_obs_arr == dobs) & (seq_num_arr == snum)
    n_pts = np.sum(mask)
    
    Zfocal_k2 = 2.0 * thx_arr[mask]
    Zfocal_k3 = 2.0 * thy_arr[mask]
    
    # Design matrix: [1, Zfocal_k2, Zfocal_k3]
    A = np.column_stack([np.ones(n_pts), Zfocal_k2, Zfocal_k3])
    
    img_params = {'day_obs': dobs, 'seq_num': snum, 'image_idx': img_idx, 'n_donuts': int(n_pts)}
    
    for j_idx, iZ in enumerate(iZs):
        # Data - model residual (model converted from meters to microns)
        resid = zk_data_matched[mask, j_idx] - zk_model_matched[mask, j_idx] * 1e6
        
        # Fit: resid = k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3
        coeffs, _, _, _ = np.linalg.lstsq(A, resid, rcond=None)
        k1, k2, k3 = coeffs
        
        img_params[f'z{iZ}_k1'] = k1
        img_params[f'z{iZ}_k2'] = k2
        img_params[f'z{iZ}_k3'] = k3
        
        # Compute fitted value for each donut in this image
        zk_fit[mask, j_idx] = k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3
    
    fit_params_list.append(img_params)

# Add fit values to matched table
aosTable_matched['zk_fit'] = list(zk_fit)

# Create fit parameters table
fit_table = QTable(fit_params_list)
print(f"Linear fit computed for {len(images)} images, {n_donuts} matched donuts")
print(f"  Field angles converted to degrees for fit (slopes in μm/deg)")
print(f"\nFit parameters table: {len(fit_table.columns)} columns")
print(f"Sample (Z4): k1 range = [{fit_table['z4_k1'].min():.3f}, {fit_table['z4_k1'].max():.3f}] μm")

<a id='fitplots'></a>
## Plot Selection

Select which day_obs values to include in fit parameter plots, residual histograms, and hexbin maps.

In [ ]:
# ============================================================
# day_obs selection for plots — edit this list to select a subset
# ============================================================
all_day_obs = sorted(set(np.array(aosTable_matched['day_obs']).tolist()))
print(f"Available day_obs values: {all_day_obs}")

# Use all days by default; modify to select a subset, e.g.:
# day_obs_plot = [20250825, 20250826]
day_obs_plot = all_day_obs

print(f"Selected day_obs for plots: {day_obs_plot}")

# Create plot mask for matched table
day_obs_matched = np.array(aosTable_matched['day_obs'])
plot_mask = np.isin(day_obs_matched, day_obs_plot)
print(f"Donuts selected for plots: {np.sum(plot_mask)} of {len(aosTable_matched)}")

# Also filter fit_table
fit_day_obs = np.array(fit_table['day_obs'])
fit_plot_mask = np.isin(fit_day_obs, day_obs_plot)
fit_table_plot = fit_table[fit_plot_mask]
print(f"Images selected for plots: {len(fit_table_plot)} of {len(fit_table)}")

In [ ]:
# Plot fit parameters k1, k2, k3 vs image index for Z4-Z15
iZs_fit_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
param_names = ['k1', 'k2', 'k3']
param_labels = ['k1 (offset)', 'k2 (x slope)', 'k3 (y slope)']
param_units = ['μm', 'μm/2deg', 'μm/2deg']
image_idx = np.arange(len(fit_table_plot))

for p_idx, (pname, plabel, punit) in enumerate(zip(param_names, param_labels, param_units)):
    fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
    fig.suptitle(f'Fit Parameter {plabel} vs Image ({date_range_str})', fontsize=14)
    
    for ax_idx, iZ in enumerate(iZs_fit_plot):
        ax = axes[ax_idx // 4, ax_idx % 4]
        vals = np.array(fit_table_plot[f'z{iZ}_{pname}'])
        ax.plot(image_idx, vals, 'o-', markersize=3)
        ax.set_title(f'Z{iZ}')
        ax.set_ylabel(punit)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
        if ax_idx // 4 == 2:
            ax.set_xlabel('Image index')
    
    plt.tight_layout()
    fig.savefig(f'{output_dir}/fit_param_{pname}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_dir}/fit_param_{pname}.png")

<a id='residhist'></a>
## Fit Residual Histograms

Histograms of per-donut residuals after subtracting the intrinsic model and the per-image linear fit
(k1 + k2 * Zfocal_k2 + k3 * Zfocal_k3) for Z4-Z15. Log scale on y-axis to highlight outlier tails.

In [ ]:
# Residual histograms: data - model*1e6 - fit, for Z4-Z15
# Uses the plot_mask to respect day_obs selection
iZs_hist = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

zk_data_plot = np.stack(aosTable_matched[f'zk_{coord_sys}'])[plot_mask]
zk_model_plot = np.stack(aosTable_matched['zk_intrinsic'])[plot_mask]
zk_fit_plot = np.stack(aosTable_matched['zk_fit'])[plot_mask]

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
fig.suptitle(f'Fit Residuals: data - model - linear fit per image ({date_range_str})', fontsize=14)

hist_range = (-1.0, 1.0)
n_bins = 100

for ax_idx, iZ in enumerate(iZs_hist):
    ax = axes[ax_idx // 4, ax_idx % 4]
    j = iZidx[iZ]
    
    resid = zk_data_plot[:, j] - zk_model_plot[:, j] * 1e6 - zk_fit_plot[:, j]
    
    # Count in-range and out-of-range
    n_total = len(resid)
    n_in = np.sum((resid >= hist_range[0]) & (resid <= hist_range[1]))
    n_out = n_total - n_in
    
    ax.hist(resid, bins=n_bins, range=hist_range, log=True, 
            edgecolor='black', linewidth=0.3, alpha=0.7)
    ax.set_title(f'Z{iZ}')
    ax.set_xlabel('μm')
    ax.set_ylabel('Count')
    ax.set_xlim(hist_range)
    
    # Annotate with std and outlier fraction
    std_val = np.std(resid)
    ax.text(0.97, 0.95, f'σ={std_val:.3f} μm\n{n_out}/{n_total} outside',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
fig.savefig(f'{output_dir}/fit_residual_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir}/fit_residual_histograms.png")

<a id='viz'></a>
## Visualizations

Create trio plots for each Zernike term: Data (linear fit subtracted), Model, and Data-Model residuals.

In [ ]:
def plot_zernike_trio(aosTable_matched, iZ, plo=4.0, phi=96.0, output_dir='.', date_range_str=''):
    """
    Create trio of plots for a Zernike term: Data, Model, Data-Model.
    
    Data is shown with per-image linear fit subtracted (constant + x,y slopes).
    
    Parameters
    ----------
    aosTable_matched : QTable
        Table with matched intra/extra donuts (must include 'zk_fit' column)
    iZ : int
        Zernike index (e.g., 4 for defocus)
    plo, phi : float
        Percentile limits for colorbar
    output_dir : str
        Directory to save figures
    date_range_str : str
        Date range string for plot title
    """
    # Set up grid for binning
    nsteps = 18 * 4 + 1
    fpradius = 1.8  # degrees
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)
    
    # Get field angles in degrees (note: x,y reversed for Rubin visualization standard)
    xval = np.rad2deg(aosTable_matched[f'thy_{coord_sys}_extra'])
    yval = np.rad2deg(aosTable_matched[f'thx_{coord_sys}_extra'])
    
    # Extract Zernike data
    zk_data_all = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    zk_fit_all = np.stack(aosTable_matched['zk_fit'])
    zk_model_all = np.stack(aosTable_matched['zk_intrinsic'])
    
    # Data with per-image linear fit subtracted (fit was on data - model*1e6)
    zval_data = zk_data_all[:, iZidx[iZ]] - zk_fit_all[:, iZidx[iZ]]
    zval_model = zk_model_all[:, iZidx[iZ]] * 1e6  # Convert meters to microns
    zval_residual = zval_data - zval_model
    
    # Create trio of plots
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Plot 1: Data (linear fit subtracted)
    mean_val_data, _, _, _ = binned_statistic_2d(
        xval, yval, zval_data,
        statistic='mean',
        bins=[xbins, ybins]
    )
    vmin_data, vmax_data = np.nanpercentile(zval_data, [plo, phi])
    im0 = axes[0].imshow(
        mean_val_data.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='viridis', interpolation='none', aspect='equal',
        vmin=vmin_data, vmax=vmax_data
    )
    add_colorbar(im0, label='μm')
    axes[0].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[0].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[0].set_title(f'Z{iZ} Data (linear fit per visit subtracted)')
    axes[0].set_aspect('equal')
    
    # Plot 2: Model
    mean_val_model, _, _, _ = binned_statistic_2d(
        xval, yval, zval_model,
        statistic='mean',
        bins=[xbins, ybins]
    )
    vmin_model, vmax_model = np.nanpercentile(zval_model, [plo, phi])
    im1 = axes[1].imshow(
        mean_val_model.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='viridis', interpolation='none', aspect='equal',
        vmin=vmin_model, vmax=vmax_model
    )
    add_colorbar(im1, label='μm')
    axes[1].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[1].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[1].set_title(f'Z{iZ} Model Intrinsic')
    axes[1].set_aspect('equal')
    
    # Plot 3: Data - Model
    mean_val_residual, _, _, _ = binned_statistic_2d(
        xval, yval, zval_residual,
        statistic='mean',
        bins=[xbins, ybins]
    )
    vmin_res, vmax_res = np.nanpercentile(zval_residual, [plo, phi])
    im2 = axes[2].imshow(
        mean_val_residual.T, origin='lower',
        extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
        cmap='RdBu_r', interpolation='none', aspect='equal',
        vmin=vmin_res, vmax=vmax_res
    )
    add_colorbar(im2, label='μm')
    axes[2].set_xlabel(f'thy_{coord_sys} [deg]')
    axes[2].set_ylabel(f'thx_{coord_sys} [deg]')
    axes[2].set_title(f'Z{iZ} Data - Model')
    axes[2].set_aspect('equal')
    
    # Overall title
    title = f'Z{iZ} Comparison: Data vs Model (matched In/Ex donuts)'
    if date_range_str:
        title += f' ({date_range_str})'
    fig.suptitle(title, fontsize=14, y=0.98)
    
    plt.tight_layout()
    
    # Save figure
    output_file = f'{output_dir}/Z{iZ}_trio_comparison.png'
    fig.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f"Saved: {output_file}")
    
    plt.show()
    
    # Print statistics
    print(f"\nZ{iZ} Statistics (μm):")
    print(f"  Data (fit-sub):   mean={np.nanmean(zval_data):7.2f}, std={np.nanstd(zval_data):7.2f}")
    print(f"  Model:            mean={np.nanmean(zval_model):7.2f}, std={np.nanstd(zval_model):7.2f}")
    print(f"  Residual:         mean={np.nanmean(zval_residual):7.2f}, std={np.nanstd(zval_residual):7.2f}")
    print("="*60)

In [ ]:
# Select matched donuts for the chosen day_obs values
aosTable_plot = aosTable_matched[plot_mask]
print(f"Donuts for plots: {len(aosTable_plot)} (day_obs: {day_obs_plot})")

# Example: Plot Z4 (defocus)
plot_zernike_trio(aosTable_plot, iZ=4, plo=plo_default, phi=phi_default,
                  output_dir=output_dir, date_range_str=date_range_str)

In [ ]:
# Plot Zernike terms Z4-Z11
iZs_to_plot = [4, 5, 6, 7, 8, 9, 10, 11]

for iZ in iZs_to_plot:
    plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                      output_dir=output_dir, date_range_str=date_range_str)

In [ ]:
# Plot Zernike terms Z12-Z22
iZs_to_plot = [iZ for iZ in iZs if 12 <= iZ <= 22]

for iZ in iZs_to_plot:
    plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                      output_dir=output_dir, date_range_str=date_range_str)